In [1]:
!pip install timm -q

import os
import torch
import timm
import numpy as np
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F

from torchvision import datasets, transforms
from torch.utils.data import DataLoader, WeightedRandomSampler
from sklearn.metrics import f1_score, classification_report

In [2]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

DATA_DIR = "/kaggle/input/datasets/veeraiahkondra/resvit/Final_Data_CLAHE"
print("Using device:", device)

Using device: cuda


In [3]:
train_transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ColorJitter(0.2,0.2),
    transforms.ToTensor(),
])

val_transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor(),
])

In [4]:
train_dataset = datasets.ImageFolder(os.path.join(DATA_DIR, "train"), transform=train_transform)
val_dataset   = datasets.ImageFolder(os.path.join(DATA_DIR, "val"), transform=val_transform)
test_dataset  = datasets.ImageFolder(os.path.join(DATA_DIR, "test"), transform=val_transform)

class_names = train_dataset.classes
print("Classes:", class_names)

# 🔥 Handle imbalance
targets = [label for _, label in train_dataset]
class_counts = np.bincount(targets)

class_weights = 1. / class_counts
sample_weights = [class_weights[t] for t in targets]

sampler = WeightedRandomSampler(sample_weights, len(sample_weights))

Classes: ['Covid-19', 'Normal', 'Pneumonia-Bacterial', 'Pneumonia-Viral']


In [5]:
train_loader = DataLoader(train_dataset, batch_size=16, sampler=sampler)
val_loader   = DataLoader(val_dataset, batch_size=16, shuffle=False)
test_loader  = DataLoader(test_dataset, batch_size=16, shuffle=False)

In [6]:
model_eff = timm.create_model('tf_efficientnet_b4', pretrained=True, num_classes=4).to(device)
model_inc = timm.create_model('inception_v3', pretrained=True, num_classes=4).to(device)

model.safetensors:   0%|          | 0.00/77.9M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/95.5M [00:00<?, ?B/s]

In [7]:

criterion = nn.CrossEntropyLoss(label_smoothing=0.1)

optimizer_eff = optim.AdamW(model_eff.parameters(), lr=3e-4)
optimizer_inc = optim.AdamW(model_inc.parameters(), lr=3e-4)

scheduler_eff = optim.lr_scheduler.CosineAnnealingLR(optimizer_eff, T_max=30)
scheduler_inc = optim.lr_scheduler.CosineAnnealingLR(optimizer_inc, T_max=30)

In [8]:
def train_effnet(epochs=30):
    best_f1 = 0

    for epoch in range(epochs):
        model_eff.train()

        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)

            optimizer_eff.zero_grad()
            outputs = model_eff(images)
            loss = criterion(outputs, labels)

            loss.backward()
            optimizer_eff.step()

        scheduler_eff.step()

        # VALIDATION
        model_eff.eval()
        preds, gts = [], []

        with torch.no_grad():
            for images, labels in val_loader:
                images = images.to(device)

                out = model_eff(images)
                pred = torch.argmax(out, dim=1)

                preds.extend(pred.cpu().numpy())
                gts.extend(labels.numpy())

        f1 = f1_score(gts, preds, average='macro')
        print(f"EffNet Epoch {epoch+1} | F1: {f1:.4f}")

        if f1 > best_f1:
            best_f1 = f1
            torch.save(model_eff.state_dict(), "best_eff.pth")

    print("✅ EfficientNet Done")

train_effnet()

EffNet Epoch 1 | F1: 0.8615
EffNet Epoch 2 | F1: 0.8702
EffNet Epoch 3 | F1: 0.8714
EffNet Epoch 4 | F1: 0.8982
EffNet Epoch 5 | F1: 0.9058
EffNet Epoch 6 | F1: 0.8992
EffNet Epoch 7 | F1: 0.9071
EffNet Epoch 8 | F1: 0.9268
EffNet Epoch 9 | F1: 0.9131
EffNet Epoch 10 | F1: 0.9135
EffNet Epoch 11 | F1: 0.9270
EffNet Epoch 12 | F1: 0.9290
EffNet Epoch 13 | F1: 0.9170
EffNet Epoch 14 | F1: 0.9505
EffNet Epoch 15 | F1: 0.9452
EffNet Epoch 16 | F1: 0.9341
EffNet Epoch 17 | F1: 0.9498
EffNet Epoch 18 | F1: 0.9395
EffNet Epoch 19 | F1: 0.9521
EffNet Epoch 20 | F1: 0.9457
EffNet Epoch 21 | F1: 0.9487
EffNet Epoch 22 | F1: 0.9487
EffNet Epoch 23 | F1: 0.9454
EffNet Epoch 24 | F1: 0.9467
EffNet Epoch 25 | F1: 0.9444
EffNet Epoch 26 | F1: 0.9492
EffNet Epoch 27 | F1: 0.9480
EffNet Epoch 28 | F1: 0.9463
EffNet Epoch 29 | F1: 0.9492
EffNet Epoch 30 | F1: 0.9492
✅ EfficientNet Done


In [9]:
def train_inception(epochs=30):
    best_f1 = 0

    for epoch in range(epochs):
        model_inc.train()

        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)

            # 🔥 resize ONLY here
            images_299 = F.interpolate(images, size=(299,299), mode='bilinear')

            optimizer_inc.zero_grad()
            outputs = model_inc(images_299)
            loss = criterion(outputs, labels)

            loss.backward()
            optimizer_inc.step()

        scheduler_inc.step()

        # VALIDATION
        model_inc.eval()
        preds, gts = [], []

        with torch.no_grad():
            for images, labels in val_loader:
                images = images.to(device)
                images_299 = F.interpolate(images, size=(299,299), mode='bilinear')

                out = model_inc(images_299)
                pred = torch.argmax(out, dim=1)

                preds.extend(pred.cpu().numpy())
                gts.extend(labels.numpy())

        f1 = f1_score(gts, preds, average='macro')
        print(f"Inception Epoch {epoch+1} | F1: {f1:.4f}")

        if f1 > best_f1:
            best_f1 = f1
            torch.save(model_inc.state_dict(), "best_inc.pth")

    print("✅ Inception Done")

train_inception()

Inception Epoch 1 | F1: 0.8372
Inception Epoch 2 | F1: 0.8642
Inception Epoch 3 | F1: 0.8468
Inception Epoch 4 | F1: 0.6235
Inception Epoch 5 | F1: 0.8707
Inception Epoch 6 | F1: 0.8834
Inception Epoch 7 | F1: 0.8935
Inception Epoch 8 | F1: 0.8951
Inception Epoch 9 | F1: 0.8970
Inception Epoch 10 | F1: 0.8979
Inception Epoch 11 | F1: 0.9206
Inception Epoch 12 | F1: 0.8892
Inception Epoch 13 | F1: 0.9077
Inception Epoch 14 | F1: 0.8952
Inception Epoch 15 | F1: 0.8938
Inception Epoch 16 | F1: 0.9238
Inception Epoch 17 | F1: 0.9191
Inception Epoch 18 | F1: 0.9266
Inception Epoch 19 | F1: 0.9322
Inception Epoch 20 | F1: 0.9374
Inception Epoch 21 | F1: 0.9349
Inception Epoch 22 | F1: 0.9410
Inception Epoch 23 | F1: 0.9389
Inception Epoch 24 | F1: 0.9424
Inception Epoch 25 | F1: 0.9435
Inception Epoch 26 | F1: 0.9419
Inception Epoch 27 | F1: 0.9485
Inception Epoch 28 | F1: 0.9460
Inception Epoch 29 | F1: 0.9454
Inception Epoch 30 | F1: 0.9473
✅ Inception Done


In [16]:
state_dict = torch.load("best_eff.pth")
print(state_dict["classifier.weight"].shape)

torch.Size([4, 1792])


In [17]:
# Fix EfficientNet
model_eff.classifier = nn.Linear(1792, 4)
model_eff.load_state_dict(torch.load("best_eff.pth", map_location=device))
model_eff.eval()

# Fix Inception
model_inc.fc = nn.Linear(model_inc.fc.in_features, 4)
if hasattr(model_inc, "AuxLogits") and model_inc.AuxLogits is not None:
    model_inc.AuxLogits.fc = nn.Linear(model_inc.AuxLogits.fc.in_features, 4)

model_inc.load_state_dict(torch.load("best_inc.pth", map_location=device))
model_inc.eval()

InceptionV3(
  (Conv2d_1a_3x3): ConvNormAct(
    (conv): Conv2d(3, 32, kernel_size=(3, 3), stride=(2, 2), bias=False)
    (bn): BatchNormAct2d(
      32, eps=0.001, momentum=0.1, affine=True, track_running_stats=True
      (drop): Identity()
      (act): ReLU(inplace=True)
    )
  )
  (Conv2d_2a_3x3): ConvNormAct(
    (conv): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), bias=False)
    (bn): BatchNormAct2d(
      32, eps=0.001, momentum=0.1, affine=True, track_running_stats=True
      (drop): Identity()
      (act): ReLU(inplace=True)
    )
  )
  (Conv2d_2b_3x3): ConvNormAct(
    (conv): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
    (bn): BatchNormAct2d(
      64, eps=0.001, momentum=0.1, affine=True, track_running_stats=True
      (drop): Identity()
      (act): ReLU(inplace=True)
    )
  )
  (Pool1): MaxPool2d(kernel_size=3, stride=2, padding=0, dilation=1, ceil_mode=False)
  (Conv2d_3b_1x1): ConvNormAct(
    (conv): Conv2d(64, 80, kernel_size

In [122]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
from sklearn.metrics import accuracy_score, f1_score
import numpy as np

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [123]:
import timm

# EfficientNet-B4
model_eff = timm.create_model("efficientnet_b4", pretrained=False, num_classes=4)
model_eff.load_state_dict(torch.load("/kaggle/working/best_eff.pth", map_location=device))
model_eff.to(device)
model_eff.eval()

# InceptionV3
model_inc = timm.create_model("inception_v3", pretrained=False, num_classes=4)
model_inc.load_state_dict(torch.load("/kaggle/working/best_inc.pth", map_location=device))
model_inc.to(device)
model_inc.eval()

InceptionV3(
  (Conv2d_1a_3x3): ConvNormAct(
    (conv): Conv2d(3, 32, kernel_size=(3, 3), stride=(2, 2), bias=False)
    (bn): BatchNormAct2d(
      32, eps=0.001, momentum=0.1, affine=True, track_running_stats=True
      (drop): Identity()
      (act): ReLU(inplace=True)
    )
  )
  (Conv2d_2a_3x3): ConvNormAct(
    (conv): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), bias=False)
    (bn): BatchNormAct2d(
      32, eps=0.001, momentum=0.1, affine=True, track_running_stats=True
      (drop): Identity()
      (act): ReLU(inplace=True)
    )
  )
  (Conv2d_2b_3x3): ConvNormAct(
    (conv): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
    (bn): BatchNormAct2d(
      64, eps=0.001, momentum=0.1, affine=True, track_running_stats=True
      (drop): Identity()
      (act): ReLU(inplace=True)
    )
  )
  (Pool1): MaxPool2d(kernel_size=3, stride=2, padding=0, dilation=1, ceil_mode=False)
  (Conv2d_3b_1x1): ConvNormAct(
    (conv): Conv2d(64, 80, kernel_size

In [124]:
test_path = "/kaggle/input/datasets/veeraiahkondra/kermany/kermany_train_test_split/test"

transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor()
])

test_dataset = datasets.ImageFolder(test_path, transform=transform)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

In [125]:
def tta(images):
    return [
        images,
        torch.flip(images, dims=[3]),
        torch.clamp(images * 1.03, 0, 1),
        torch.clamp(images * 0.97, 0, 1),
    ]

In [126]:
def ensemble_predict(images):

    images = images.to(device)
    img299 = F.interpolate(images, size=(299,299))

    p1 = torch.softmax(model_eff(images), dim=1)
    p2 = torch.softmax(model_inc(img299), dim=1)

    pf = 0.7*p1 + 0.3*p2

    boost = torch.tensor([1.0, 1.0, 1.5, 1.5], device=device).unsqueeze(0)

    pf = pf * boost
    pf = pf / pf.sum(dim=1, keepdim=True)

    return pf

In [127]:
def tta_predict(images):

    outputs = []

    for v in tta(images):
        out = ensemble_predict(v)
        outputs.append(out)

    return torch.stack(outputs).mean(0)

In [133]:
print(test_dataset.class_to_idx)

{'normal': 0, 'pneumonia': 1}


In [137]:
from sklearn.metrics import accuracy_score, f1_score

model_eff.eval()
model_inc.eval()

y_true = []
y_pred = []

with torch.no_grad():

    for images, labels in test_loader:

        images = images.to(device)

        # 4-class prediction
        outputs = tta_predict(images)

        # 🔥 Convert 4 → 2 classes
        normal = outputs[:, 0]
        pneumonia = outputs[:, 1] + outputs[:, 2]

        outputs_bin = torch.stack([normal, pneumonia], dim=1)

        preds = torch.argmax(outputs_bin, dim=1)

        # store
        y_true.extend(labels.cpu().numpy())
        y_pred.extend(preds.cpu().numpy())

# metrics
acc = accuracy_score(y_true, y_pred)
f1  = f1_score(y_true, y_pred, average="macro")

print("🔥 Kermany BEFORE Fine-Tuning")
print("Accuracy:", acc)
print("Macro F1:", f1)

🔥 Kermany BEFORE Fine-Tuning
Accuracy: 0.8352159468438538
Macro F1: 0.8065897530500823


In [138]:
print(model_eff.classifier.weight.shape)

torch.Size([4, 1792])


In [140]:
train_path = "/kaggle/input/datasets/veeraiahkondra/kermany/kermany_train_test_split/train"
val_path   = "/kaggle/input/datasets/veeraiahkondra/kermany/kermany_train_test_split/val"

train_dataset = datasets.ImageFolder(train_path, transform=transform)
val_dataset   = datasets.ImageFolder(val_path, transform=transform)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader   = DataLoader(val_dataset, batch_size=32, shuffle=False)

In [141]:
# ==============================
# 🔥 FREEZE BACKBONES
# ==============================

for param in model_eff.parameters():
    param.requires_grad = False

for param in model_inc.parameters():
    param.requires_grad = False

# Unfreeze classifier layers only
for param in model_eff.classifier.parameters():
    param.requires_grad = True

for param in model_inc.fc.parameters():
    param.requires_grad = True


# ==============================
# 🔥 OPTIMIZER
# ==============================

optimizer = torch.optim.Adam(
    list(model_eff.classifier.parameters()) +
    list(model_inc.fc.parameters()),
    lr=1e-5
)

criterion = nn.CrossEntropyLoss()

print("🔥 Fine-tuning started...")


# ==============================
# 🔥 TRAINING
# ==============================

for epoch in range(10):   # 10 is safer than 20

    model_eff.train()
    model_inc.train()

    total_loss = 0

    for images, labels in train_loader:

        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        # Inception input
        img299 = F.interpolate(
            images,
            size=(299,299),
            mode='bilinear',
            align_corners=False
        )

        # 4-class logits
        logits1 = model_eff(images)
        logits2 = model_inc(img299)

        # ====================================
        # Convert 4-class → binary logits
        # ====================================

        # EfficientNet
        normal1 = logits1[:,0]
        pneumonia1 = logits1[:,1] + logits1[:,2]

        # Inception
        normal2 = logits2[:,0]
        pneumonia2 = logits2[:,1] + logits2[:,2]

        logits_eff = torch.stack([normal1, pneumonia1], dim=1)
        logits_inc = torch.stack([normal2, pneumonia2], dim=1)

        # Weighted fusion
        logits = 0.7 * logits_eff + 0.3 * logits_inc

        # Loss
        loss = criterion(logits, labels)

        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(f"Epoch {epoch+1}/10 | Loss: {total_loss:.4f}")

🔥 Fine-tuning started...
Epoch 1/10 | Loss: 107.2885
Epoch 2/10 | Loss: 93.8076
Epoch 3/10 | Loss: 88.6371
Epoch 4/10 | Loss: 88.5108
Epoch 5/10 | Loss: 86.8625
Epoch 6/10 | Loss: 85.9956
Epoch 7/10 | Loss: 83.4617
Epoch 8/10 | Loss: 85.7489
Epoch 9/10 | Loss: 84.0383
Epoch 10/10 | Loss: 82.1914


In [142]:
from sklearn.metrics import accuracy_score, f1_score

model_eff.eval()
model_inc.eval()

y_true = []
y_pred = []

with torch.no_grad():

    for images, labels in test_loader:

        images = images.to(device)

        # ====================================
        # TTA prediction
        # ====================================

        outputs = tta_predict(images)   # (B,4)

        # ====================================
        # Convert 4-class → binary
        # ====================================

        normal = outputs[:,0]
        pneumonia = outputs[:,1] + outputs[:,2]

        outputs_bin = torch.stack(
            [normal, pneumonia],
            dim=1
        )

        preds = torch.argmax(outputs_bin, dim=1)

        y_true.extend(labels.cpu().numpy())
        y_pred.extend(preds.cpu().numpy())


# ====================================
# Metrics
# ====================================

acc = accuracy_score(y_true, y_pred)
f1  = f1_score(y_true, y_pred, average="macro")

print("\n🔥 Kermany AFTER Fine-Tuning")
print("Accuracy:", acc)
print("Macro F1:", f1)


🔥 Kermany AFTER Fine-Tuning
Accuracy: 0.8777408637873754
Macro F1: 0.8761418813266397


In [145]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import transforms
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import accuracy_score, f1_score
from PIL import Image
import numpy as np
import os
import timm

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [144]:
model_eff = timm.create_model("efficientnet_b4", pretrained=False, num_classes=4)
model_eff.load_state_dict(torch.load("/kaggle/working/best_eff.pth", map_location=device))
model_eff.to(device).eval()

model_inc = timm.create_model("inception_v3", pretrained=False, num_classes=4)
model_inc.load_state_dict(torch.load("/kaggle/working/best_inc.pth", map_location=device))
model_inc.to(device).eval()

InceptionV3(
  (Conv2d_1a_3x3): ConvNormAct(
    (conv): Conv2d(3, 32, kernel_size=(3, 3), stride=(2, 2), bias=False)
    (bn): BatchNormAct2d(
      32, eps=0.001, momentum=0.1, affine=True, track_running_stats=True
      (drop): Identity()
      (act): ReLU(inplace=True)
    )
  )
  (Conv2d_2a_3x3): ConvNormAct(
    (conv): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), bias=False)
    (bn): BatchNormAct2d(
      32, eps=0.001, momentum=0.1, affine=True, track_running_stats=True
      (drop): Identity()
      (act): ReLU(inplace=True)
    )
  )
  (Conv2d_2b_3x3): ConvNormAct(
    (conv): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
    (bn): BatchNormAct2d(
      64, eps=0.001, momentum=0.1, affine=True, track_running_stats=True
      (drop): Identity()
      (act): ReLU(inplace=True)
    )
  )
  (Pool1): MaxPool2d(kernel_size=3, stride=2, padding=0, dilation=1, ceil_mode=False)
  (Conv2d_3b_1x1): ConvNormAct(
    (conv): Conv2d(64, 80, kernel_size

In [143]:
import os
from PIL import Image

import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

# ====================================
# PATH
# ====================================

covid_path = "/kaggle/input/datasets/veeraiahkondra/covid19/COVID-19_Radiography_Dataset"

# ====================================
# TRANSFORM
# ====================================

transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor(),
])

# ====================================
# DATASET
# Normal -> 0
# COVID + Viral -> 1
# ====================================

class COVIDDataset(Dataset):

    def __init__(self, root_dir, transform=None):

        self.samples = []
        self.transform = transform

        mapping = {
            "Normal": 0,
            "COVID": 1,
            "Viral Pneumonia": 1
        }

        for cls in mapping:

            folder = os.path.join(root_dir, cls)

            for img_name in os.listdir(folder):

                img_path = os.path.join(folder, img_name)

                self.samples.append(
                    (img_path, mapping[cls])
                )

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):

        img_path, label = self.samples[idx]

        image = Image.open(img_path).convert("RGB")

        if self.transform:
            image = self.transform(image)

        return image, label


# ====================================
# DATASET + LOADER
# ====================================

covid_dataset = COVIDDataset(
    covid_path,
    transform=transform
)

covid_loader = DataLoader(
    covid_dataset,
    batch_size=32,
    shuffle=False
)

print("✅ COVID dataset loaded")
print("Samples:", len(covid_dataset))

✅ COVID dataset loaded
Samples: 15153


In [146]:
import torch.nn.functional as F

# ====================================
# TTA
# ====================================

def tta_transforms(images):

    t1 = images

    t2 = torch.flip(images, dims=[3])

    t3 = torch.clamp(images * 1.03, 0, 1)

    t4 = torch.clamp(images * 0.97, 0, 1)

    return [t1, t2, t3, t4]

In [147]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model_eff.to(device)
model_inc.to(device)

model_eff.eval()
model_inc.eval()

# ====================================
# ENSEMBLE
# ====================================

def ensemble_predict(images):

    images = images.to(device)

    img299 = F.interpolate(
        images,
        size=(299,299),
        mode='bilinear',
        align_corners=False
    )

    logits1 = model_eff(images)
    logits2 = model_inc(img299)

    p1 = torch.softmax(logits1, dim=1)
    p2 = torch.softmax(logits2, dim=1)

    # weighted fusion
    pfusion = 0.7 * p1 + 0.3 * p2

    # class boost
    boost = torch.tensor(
        [1.0, 1.0, 1.5, 1.5],
        device=device
    ).unsqueeze(0)

    pfusion = pfusion * boost

    pfusion = pfusion / pfusion.sum(
        dim=1,
        keepdim=True
    )

    return pfusion

In [148]:
# ====================================
# TTA prediction
# ====================================

def tta_predict(images):

    views = tta_transforms(images)

    outputs = []

    for v in views:

        out = ensemble_predict(v)

        outputs.append(out)

    outputs = torch.stack(outputs)

    return outputs.mean(dim=0)

In [149]:
from sklearn.metrics import accuracy_score, f1_score

model_eff.eval()
model_inc.eval()

y_true = []
y_pred = []

with torch.no_grad():

    for images, labels in covid_loader:

        images = images.to(device)

        outputs = tta_predict(images)   # (B,4)

        # ====================================
        # Convert 4-class -> binary
        # ====================================

        normal = outputs[:,0]

        covid = outputs[:,2] + outputs[:,3]

        outputs_bin = torch.stack(
            [normal, covid],
            dim=1
        )

        preds = torch.argmax(outputs_bin, dim=1)

        y_true.extend(labels.numpy())
        y_pred.extend(preds.cpu().numpy())


# ====================================
# Metrics
# ====================================

acc = accuracy_score(y_true, y_pred)

f1 = f1_score(
    y_true,
    y_pred,
    average="macro"
)

print("\n🔥 COVID BEFORE Fine-Tuning")
print("Accuracy:", acc)
print("Macro F1:", f1)


🔥 COVID BEFORE Fine-Tuning
Accuracy: 0.6739259552563849
Macro F1: 0.5693169246383728


In [159]:
# =========================================================
# 🚀 COVID AFTER FINETUNING ONLY
# =========================================================

# ---------------------------------------------
# Freeze all layers
# ---------------------------------------------
for param in model_eff.parameters():
    param.requires_grad = False

for param in model_inc.parameters():
    param.requires_grad = False

# ---------------------------------------------
# Train only classifier heads
# ---------------------------------------------
for param in model_eff.classifier.parameters():
    param.requires_grad = True

for param in model_inc.fc.parameters():
    param.requires_grad = True

# ---------------------------------------------
# Optimizer
# ---------------------------------------------
optimizer = torch.optim.Adam(

    list(model_eff.classifier.parameters()) +
    list(model_inc.fc.parameters()),

    lr=1e-4
)

criterion = nn.CrossEntropyLoss()

# =========================================================
# 🚀 FINETUNING
# =========================================================

best_f1 = 0

print("🔥 COVID Fine-Tuning Started...\n")

for epoch in range(5):

    model_eff.train()
    model_inc.train()

    total_loss = 0

    for images, labels in covid_loader:

        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        # Inception resize
        img299 = F.interpolate(
            images,
            size=(299,299),
            mode='bilinear',
            align_corners=False
        )

        # Forward
        logits1 = model_eff(images)
        logits2 = model_inc(img299)

        # -------------------------------------
        # 4-class → binary conversion
        # class0 = Normal
        # class2 = COVID
        # -------------------------------------

        normal1 = logits1[:,0]
        covid1  = logits1[:,2]

        normal2 = logits2[:,0]
        covid2  = logits2[:,2]

        logits_eff = torch.stack(
            [normal1, covid1],
            dim=1
        )

        logits_inc = torch.stack(
            [normal2, covid2],
            dim=1
        )

        # Weighted fusion
        logits = 0.7 * logits_eff + 0.3 * logits_inc

        loss = criterion(logits, labels)

        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    # =====================================================
    # VALIDATION
    # =====================================================

    model_eff.eval()
    model_inc.eval()

    y_true = []
    y_pred = []

    with torch.no_grad():

        for images, labels in covid_loader:

            images = images.to(device)

            # =========================
            # TTA
            # =========================

            views = tta_transforms(images)

            outputs_all = []

            for v in views:

                img299 = F.interpolate(
                    v,
                    size=(299,299),
                    mode='bilinear',
                    align_corners=False
                )

                out1 = torch.softmax(model_eff(v), dim=1)
                out2 = torch.softmax(model_inc(img299), dim=1)

                out = 0.7 * out1 + 0.3 * out2

                outputs_all.append(out)

            outputs = torch.stack(outputs_all).mean(dim=0)

            # binary mapping
            normal = outputs[:,0]
            covid = outputs[:,2]

            outputs_bin = torch.stack(
                [normal, covid],
                dim=1
            )

            preds = torch.argmax(outputs_bin, dim=1)

            y_true.extend(labels.numpy())
            y_pred.extend(preds.cpu().numpy())

    acc = accuracy_score(y_true, y_pred)

    f1 = f1_score(
        y_true,
        y_pred,
        average="macro"
    )

    print(
        f"Epoch {epoch+1}/5 | "
        f"Loss: {total_loss:.4f} | "
        f"Acc: {acc:.4f} | "
        f"Macro F1: {f1:.4f}"
    )

    # Save best
    if f1 > best_f1:

        best_f1 = f1

        torch.save(model_eff.state_dict(), "covid_best_eff.pth")
        torch.save(model_inc.state_dict(), "covid_best_inc.pth")

        print("✅ Best model saved")

# =========================================================
# 🚀 FINAL AFTER FINETUNING RESULT
# =========================================================

print("\n🔥 COVID AFTER Fine-Tuning")
print("Best Macro F1:", best_f1)

🔥 COVID Fine-Tuning Started...



KeyboardInterrupt: 